# Administrative boundaries — catalog explorer (offline)

Before fetching anything, it helps to know **which datasets exist** and **which
selector each needs**. This notebook is fully offline — it only reads the bundled
catalog. By the end you will be able to list every admin dataset id and read its
provider, ADM level / layer, native CRS, and required selectors.

## Setup

The admin `Catalog` loads the bundled per-provider YAML files.

In [1]:
import pandas as pd

from earthlens.admin import Catalog

cat = Catalog()
len(cat)

2026-06-28 16:47:46 | INFO | pyramids.base.config | Logging is configured.


15

## Every dataset, as a table

Each row carries its `provider`, the ADM level (geoBoundaries / CGAZ) or layer
(Natural Earth / TIGER), the native CRS, and the selectors the request must supply.

In [2]:
rows = [
    {
        'id': d.id,
        'provider': d.provider,
        'adm_level': d.adm_level,
        'layer': d.layer,
        'native_crs': d.native_crs,
        'required_selectors': ', '.join(d.required_selectors) or '—',
    }
    for d in cat.datasets.values()
]
pd.DataFrame(rows).sort_values('id').reset_index(drop=True)

,id,provider,adm_level,layer,native_crs,required_selectors
0,cgaz:adm0,cgaz,ADM0,None,undefined,—
1,cgaz:adm1,cgaz,ADM1,None,undefined,—
2,cgaz:adm2,cgaz,ADM2,None,undefined,—
3,geoboundaries:adm0,geoboundaries,ADM0,None,EPSG:4326,country
4,geoboundaries:adm1,geoboundaries,ADM1,None,EPSG:4326,country
5,geoboundaries:adm2,geoboundaries,ADM2,None,EPSG:4326,country
6,geoboundaries:adm3,geoboundaries,ADM3,None,EPSG:4326,country
7,geoboundaries:adm4,geoboundaries,ADM4,None,EPSG:4326,country
8,geoboundaries:adm5,geoboundaries,ADM5,None,EPSG:4326,country
9,natural_earth:countries,natural_earth,None,admin_0_countries,EPSG:4326,—


## Datasets per provider

Four providers, each with a different addressing model — per-country ADM levels
(geoBoundaries), seamless global levels (CGAZ), scaled cultural layers
(Natural Earth), and US entities (TIGER).

In [3]:
by_provider = {}
for d in cat.datasets.values():
    by_provider.setdefault(d.provider, []).append(d.id)
for provider, ids in sorted(by_provider.items()):
    print(f'{provider:14} {len(ids):>2}  {sorted(ids)}')

cgaz            3  ['cgaz:adm0', 'cgaz:adm1', 'cgaz:adm2']
geoboundaries   6  ['geoboundaries:adm0', 'geoboundaries:adm1', 'geoboundaries:adm2', 'geoboundaries:adm3', 'geoboundaries:adm4', 'geoboundaries:adm5']
natural_earth   2  ['natural_earth:countries', 'natural_earth:states']
tiger           4  ['tiger:county', 'tiger:nation', 'tiger:state', 'tiger:tract']


## Look up one dataset

`Catalog.get(id)` returns the typed row; an unknown id raises a clear `ValueError`
with a did-you-mean hint.

In [4]:
row = cat.get('tiger:tract')
print('provider :', row.provider)
print('layer    :', row.layer)
print('per_state:', row.per_state)
print('selectors:', row.required_selectors)
print('license  :', row.license_note)

provider : tiger
layer    : tract
per_state: True
selectors: ('state',)
license  : U.S. Census Bureau TIGER/Line cartographic boundaries, public domain.


A typo is rejected rather than silently returning nothing:

In [5]:
try:
    cat.get('geoboundaries:adm9')
except ValueError as exc:
    print(exc)

'geoboundaries:adm9' is not in the admin boundary catalog. Known datasets: ['cgaz:adm0', 'cgaz:adm1', 'cgaz:adm2', 'geoboundaries:adm0', 'geoboundaries:adm1', 'geoboundaries:adm2', 'geoboundaries:adm3', 'geoboundaries:adm4', 'geoboundaries:adm5', 'natural_earth:countries', 'natural_earth:states', 'tiger:county', 'tiger:nation', 'tiger:state', 'tiger:tract']. Did you mean 'geoboundaries:adm5'?


## Takeaway

- `Catalog()` lists 15 datasets across geoBoundaries, CGAZ, Natural Earth, TIGER.
- Each row tells you the selector to pass (`country=` / `scale=` / `year=` /
  `state=`) and the native CRS (the backend normalises every result to EPSG:4326).
- **GADM is intentionally absent** — its no-commercial / no-redistribute license is
  incompatible; the four shipped sources are CC-BY-4.0 or public domain.